## Currency Conversion

In [1]:
!pip install -q langchain-google-genai google-generativeai langchain-core requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.8 MB/s eta 0:00:00


In [18]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests
from langchain_core.tools import InjectedToolArg
from typing import Annotated

In [19]:
# Tool creation

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/15f0774e6408b7c15009e7d3/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value * conversion_rate


In [20]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'integer'}}

In [21]:
convert.invoke({'base_currency_value':10, 'conversion_rate':85.16})

851.5999999999999

In [22]:
# Tool binding

API_KEY = "AQ#######################"

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    google_api_key=API_KEY
)

llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [23]:
human_msg = HumanMessage('What is the conversion factor between USD and INR, and based on that can you convert 10 usd to inr')

In [24]:
messages = [human_msg]

In [25]:
# Tool Calling
ai_message = llm_with_tools.invoke(messages)

In [26]:
messages.append(ai_message)

In [27]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'target_currency': 'INR', 'base_currency': 'USD'},
  'id': '5cdcca55-f8a4-403d-ae23-56c5c824295a',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10},
  'id': 'c54e3066-7920-48e7-bedb-cc9fd98c59ec',
  'type': 'tool_call'}]

In [28]:
import json

for tool_call in ai_message.tool_calls:

  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    messages.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1

  if tool_call['name'] == 'convert':
    # fetch the current arg
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)

In [29]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR, and based on that can you convert 10 usd to inr', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'convert', 'arguments': '{"base_currency_value": 10}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019eb575-0e40-7680-a64e-1cdf0101daf4-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'target_currency': 'INR', 'base_currency': 'USD'}, 'id': '5cdcca55-f8a4-403d-ae23-56c5c824295a', 'type': 'tool_call'}, {'name': 'convert', 'args': {'base_currency_value': 10, 'conversion_rate': 95.369}, 'id': 'c54e3066-7920-48e7-bedb-cc9fd98c59ec', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 150, 'output_tokens': 44, 'total_tokens': 194, 'input_token_details': {'cache_read': 0}}),
 ToolMessage(content='{"result": "succes

In [30]:
llm_with_tools.invoke(messages).content

'The conversion factor between USD and INR is 95.369.\n10 USD is equal to 953.69 INR.'